# 01 — Exploratory Data Analysis of Garmin Health Metrics

This notebook explores Garmin Connect data exported from several sources:

- Body Battery
- Activities
- Heart Rate
- Sleep
- Stress

The goal is to understand daily physiological patterns related to recovery, sleep quality, training load, stress, and heart rate.

This analysis is exploratory and personal. It is not a medical diagnosis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

### Notes

On importe les bibliothèques principales.  
`pandas` sert à manipuler les tableaux, `numpy` aux calculs numériques, `matplotlib` aux graphiques, et `Path` permet de gérer proprement les chemins de fichiers.

In [4]:
DATA_DIR = Path("./data")

paths = {
    "activities": DATA_DIR / "Activities.csv",
    "hr": DATA_DIR / "HR.csv",
    "sleep": DATA_DIR / "Sleep.csv",
    "stress": DATA_DIR / "Stress.csv",
    "body_battery": DATA_DIR / "body_battery.csv",
}

for name, path in paths.items():
    print(f"{name}: {path} -> exists = {path.exists()}")

activities: data\Activities.csv -> exists = True
hr: data\HR.csv -> exists = True
sleep: data\Sleep.csv -> exists = True
stress: data\Stress.csv -> exists = True
body_battery: data\body_battery.csv -> exists = True


In [5]:
activities_raw = pd.read_csv(paths["activities"])
hr_raw = pd.read_csv(paths["hr"])
sleep_raw = pd.read_csv(paths["sleep"])
stress_raw = pd.read_csv(paths["stress"])
body_battery_raw = pd.read_csv(paths["body_battery"])

raw_datasets = {
    "activities": activities_raw,
    "hr": hr_raw,
    "sleep": sleep_raw,
    "stress": stress_raw,
    "body_battery": body_battery_raw,
}

for name, df in raw_datasets.items():
    print(f"\n{name.upper()}")
    print(f"Shape: {df.shape}")
    display(df.head())


ACTIVITIES
Shape: (168, 45)


,Activity Type,Date,Favorite,Title,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,Avg Cadence,Max Cadence,Avg Pace,Best Pace,Total Ascent,Total Descent,Avg Stride Length,Avg Vertical Ratio,Avg Vertical Oscillation,Avg Ground Contact Time,Avg GCT Balance,Normalized Power® (NP®),Training Stress Score®,Avg Power,Max Power,Steps,Total Reps,Total Sets,Body Battery Drain,Decompression,Best Lap Time,Number of Laps,Max Temp,Avg Resp,Min Resp,Max Resp,Stress Change,Stress Start,Stress End,Avg Stress,Max Stress,Moving Time,Elapsed Time,Min Elevation,Max Elevation
0,Walking,2026-05-24 19:07:08,False,Paris Walking,0.81,47,00:10:37,110,119,0.3,106,122,13:04,9:49,8,--,0.72,--,--,--,--,--,0.0,--,--,"1,156",--,--,-1,No,00:10:37,1,--,--,--,--,--,--,--,--,--,00:10:17,00:10:37,36,49
1,Walking,2026-05-24 17:50:04,False,Paris Walking,0.85,63,00:16:17,103,119,0.4,71,150,19:13,6:59,5,11,0.73,--,--,--,--,--,0.0,--,--,"1,292",--,--,-1,No,00:16:17,1,--,--,--,--,--,--,--,--,--,00:11:06,00:16:17,36,49
2,Walking,2026-05-24 12:39:14,False,Paris Walking,0.51,36,00:10:40,93,111,0.2,52,158,20:47,7:34,10,8,0.93,--,--,--,--,--,0.0,--,--,796,--,--,--,No,00:10:40,1,--,--,--,--,--,--,--,--,--,00:07:42,00:10:40,38,45
3,Walking,2026-05-23 10:25:26,False,Vélizy-Villacoublay Walking,0.67,60,00:17:45,95,119,0.2,50,158,26:25,8:05,3,3,0.75,--,--,--,--,--,0.0,--,--,"1,100",--,--,-1,No,00:17:45,1,--,--,--,--,--,--,--,--,--,00:11:15,00:17:45,174,181
4,Walking,2026-05-21 19:00:04,False,Paris Walking,0.36,32,00:05:14.3,129,150,0.6,85,122,14:43,6:03,5,5,0.80,--,--,--,--,--,0.0,--,--,498,--,--,--,No,00:05:14.3,1,--,--,--,--,--,--,--,--,--,00:04:50,00:05:14.3,29,35



HR
Shape: (149, 3)


,Date,Resting,High
0,NaN,NaN,NaN
1,May 25,64 bpm,89 bpm
2,May 24,67 bpm,131 bpm
3,May 23,63 bpm,134 bpm
4,May 22,63 bpm,128 bpm



SLEEP
Shape: (152, 7)


,Date,Score,Quality,Duration,Sleep Need,Bedtime,Wake Time
0,May 25,38,Poor,3h 17m,9h 0m,11:31 AM,2:49 PM
1,May 24,42,Poor,4h 47m,7h 0m,4:00 AM,9:39 AM
2,May 23,78,Fair,6h 30m,8h 20m,2:53 AM,9:53 AM
3,May 22,78,Fair,10h 23m,7h 10m,10:56 PM,9:56 AM
4,May 21,65,Fair,8h 49m,8h 20m,12:35 AM,9:54 AM



STRESS
Shape: (147, 6)


,Date,Avg,Rest,Low,Medium Stress,High Stress
0,May 25,17,14h 8min,38min,2min,--
1,May 24,40,8h 38min,5h 36min,4h 59min,2h 0min
2,May 23,40,8h 1min,5h 49min,3h 47min,2h 14min
3,May 22,36,7h 14min,9h 27min,3h 59min,32min
4,May 21,40,6h 54min,6h 48min,4h 25min,1h 30min



BODY_BATTERY
Shape: (153, 9)


,date,daily_high,daily_low,daily_range,source_chart,extraction_confidence,notes,high_source,low_source
0,2025-12-24,23.0,8.0,15.0,Dec09-Jan05,high,NaN,blue dot,black dot
1,2025-12-25,42.0,6.0,36.0,Dec09-Jan05,high,NaN,blue dot,black dot
2,2025-12-26,79.0,16.0,63.0,Dec09-Jan05,high,NaN,blue dot,black dot
3,2025-12-27,68.0,18.0,50.0,Dec09-Jan05,high,NaN,blue dot,black dot
4,2025-12-28,68.0,17.0,51.0,Dec09-Jan05,high,NaN,blue dot,black dot


In [6]:
for name, df in raw_datasets.items():
    print("=" * 80)
    print(name.upper())
    print("=" * 80)
    print(df.info())
    print("\nMissing values per column:")
    display(df.isna().sum())

ACTIVITIES
<class 'pandas.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 45 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Activity Type             168 non-null    str    
 1   Date                      168 non-null    str    
 2   Favorite                  168 non-null    bool   
 3   Title                     168 non-null    str    
 4   Distance                  168 non-null    str    
 5   Calories                  168 non-null    str    
 6   Time                      168 non-null    str    
 7   Avg HR                    168 non-null    str    
 8   Max HR                    168 non-null    int64  
 9   Aerobic TE                168 non-null    float64
 10  Avg Cadence               168 non-null    str    
 11  Max Cadence               168 non-null    str    
 12  Avg Pace                  168 non-null    str    
 13  Best Pace                 168 non-null    str    
 14  Total Asce

Activity Type               0
Date                        0
Favorite                    0
Title                       0
Distance                    0
Calories                    0
Time                        0
Avg HR                      0
Max HR                      0
Aerobic TE                  0
Avg Cadence                 0
Max Cadence                 0
Avg Pace                    0
Best Pace                   0
Total Ascent                0
Total Descent               0
Avg Stride Length           0
Avg Vertical Ratio          0
Avg Vertical Oscillation    0
Avg Ground Contact Time     0
Avg GCT Balance             0
Normalized Power® (NP®)     0
Training Stress Score®      0
Avg Power                   0
Max Power                   0
Steps                       0
Total Reps                  0
Total Sets                  0
Body Battery Drain          0
Decompression               0
Best Lap Time               0
Number of Laps              0
Max Temp                    0
Avg Resp  

HR
<class 'pandas.DataFrame'>
RangeIndex: 149 entries, 0 to 148
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   Date     148 non-null    str  
 1   Resting  148 non-null    str  
 2   High     148 non-null    str  
dtypes: str(3)
memory usage: 3.6 KB
None

Missing values per column:


Date       1
Resting    1
High       1
dtype: int64

SLEEP
<class 'pandas.DataFrame'>
RangeIndex: 152 entries, 0 to 151
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Date        152 non-null    str  
 1   Score       152 non-null    str  
 2   Quality     152 non-null    str  
 3   Duration    152 non-null    str  
 4   Sleep Need  152 non-null    str  
 5   Bedtime     152 non-null    str  
 6   Wake Time   152 non-null    str  
dtypes: str(7)
memory usage: 8.4 KB
None

Missing values per column:


Date          0
Score         0
Quality       0
Duration      0
Sleep Need    0
Bedtime       0
Wake Time     0
dtype: int64

STRESS
<class 'pandas.DataFrame'>
RangeIndex: 147 entries, 0 to 146
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Date           147 non-null    str  
 1   Avg            147 non-null    int64
 2   Rest           147 non-null    str  
 3   Low            147 non-null    str  
 4   Medium Stress  147 non-null    str  
 5   High Stress    147 non-null    str  
dtypes: int64(1), str(5)
memory usage: 7.0 KB
None

Missing values per column:


Date             0
Avg              0
Rest             0
Low              0
Medium Stress    0
High Stress      0
dtype: int64

BODY_BATTERY
<class 'pandas.DataFrame'>
RangeIndex: 153 entries, 0 to 152
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   153 non-null    str    
 1   daily_high             148 non-null    float64
 2   daily_low              148 non-null    float64
 3   daily_range            148 non-null    float64
 4   source_chart           153 non-null    str    
 5   extraction_confidence  148 non-null    str    
 6   notes                  5 non-null      str    
 7   high_source            148 non-null    str    
 8   low_source             148 non-null    str    
dtypes: float64(3), str(6)
memory usage: 10.9 KB
None

Missing values per column:


date                       0
daily_high                 5
daily_low                  5
daily_range                5
source_chart               0
extraction_confidence      5
notes                    148
high_source                5
low_source                 5
dtype: int64

### Notes

On inspecte la structure de chaque tableau : colonnes, types de données et valeurs manquantes.

À ce stade, beaucoup de colonnes sont encore en texte, par exemple `64 bpm`, `3h 17m`, ou `00:10:37`.  
Il faudra les convertir en valeurs numériques pour pouvoir faire des graphiques et du machine learning.

In [7]:
def clean_numeric(value):
    """
    Convert Garmin-style values to numeric floats.

    Examples:
    - "64 bpm" -> 64
    - "1,156" -> 1156
    - "--" -> NaN
    - "0.81" -> 0.81
    """
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip()
    
    if value in ["--", ""]:
        return np.nan
    
    value = value.replace(",", "")
    value = value.replace("bpm", "")
    value = value.strip()
    
    try:
        return float(value)
    except ValueError:
        return np.nan


def duration_to_minutes(value):
    """
    Convert Garmin duration strings to minutes.

    Examples:
    - "3h 17m" -> 197
    - "14h 8min" -> 848
    - "38min" -> 38
    - "--" -> 0
    - "00:10:37" -> 10.62
    """
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip()
    
    if value in ["--", ""]:
        return 0
    
    # Case: HH:MM:SS
    if ":" in value:
        parts = value.split(":")
        if len(parts) == 3:
            h, m, s = map(float, parts)
            return h * 60 + m + s / 60
        elif len(parts) == 2:
            m, s = map(float, parts)
            return m + s / 60
    
    value = value.replace("min", "m")
    
    hours = 0
    minutes = 0
    
    if "h" in value:
        h_part = value.split("h")[0].strip()
        hours = float(h_part) if h_part else 0
        value = value.split("h")[1].strip()
    
    if "m" in value:
        m_part = value.split("m")[0].strip()
        minutes = float(m_part) if m_part else 0
    
    return hours * 60 + minutes


def parse_garmin_short_date(value):
    """
    Convert dates like 'May 25' or 'Dec 24' into full dates.

    The dataset crosses from Dec 2025 to May 2026.
    So:
    - Dec -> 2025
    - Jan to May -> 2026
    """
    if pd.isna(value):
        return pd.NaT
    
    value = str(value).strip()
    
    if value == "":
        return pd.NaT
    
    month = value.split()[0]
    year = 2025 if month == "Dec" else 2026
    
    return pd.to_datetime(f"{value} {year}", format="%b %d %Y", errors="coerce")

### Notes

On crée des fonctions de nettoyage réutilisables.

Les exports Garmin mélangent souvent des nombres avec du texte, par exemple `64 bpm` ou `3h 17m`.  
Pour analyser les données, on doit transformer ces valeurs en nombres exploitables.

La fonction de date est spécifique à ce dataset, car les données vont de décembre 2025 à mai 2026.

In [8]:
hr = hr_raw.copy()

hr["date"] = hr["Date"].apply(parse_garmin_short_date)
hr["resting_hr"] = hr["Resting"].apply(clean_numeric)
hr["high_hr"] = hr["High"].apply(clean_numeric)

hr = hr[["date", "resting_hr", "high_hr"]]
hr = hr.dropna(subset=["date"])
hr = hr.sort_values("date").reset_index(drop=True)

display(hr.head())
display(hr.tail())

,date,resting_hr,high_hr
0,2025-12-24,88.0,105.0
1,2025-12-25,68.0,125.0
2,2025-12-26,66.0,109.0
3,2025-12-27,68.0,139.0
4,2025-12-28,69.0,137.0


,date,resting_hr,high_hr
143,2026-05-21,64.0,142.0
144,2026-05-22,63.0,128.0
145,2026-05-23,63.0,134.0
146,2026-05-24,67.0,131.0
147,2026-05-25,64.0,89.0


### Notes

On nettoie les données de fréquence cardiaque.

On garde trois informations principales :
- la date
- la fréquence cardiaque au repos
- la fréquence cardiaque maximale de la journée

La fréquence cardiaque au repos sera très importante pour étudier la récupération et la fatigue.

In [9]:
sleep = sleep_raw.copy()

sleep["date"] = sleep["Date"].apply(parse_garmin_short_date)
sleep["sleep_score"] = sleep["Score"].apply(clean_numeric)
sleep["sleep_duration_min"] = sleep["Duration"].apply(duration_to_minutes)
sleep["sleep_need_min"] = sleep["Sleep Need"].apply(duration_to_minutes)

sleep["sleep_duration_h"] = sleep["sleep_duration_min"] / 60
sleep["sleep_need_h"] = sleep["sleep_need_min"] / 60
sleep["sleep_debt_h"] = sleep["sleep_need_h"] - sleep["sleep_duration_h"]

sleep = sleep[
    [
        "date",
        "sleep_score",
        "Quality",
        "sleep_duration_min",
        "sleep_duration_h",
        "sleep_need_min",
        "sleep_need_h",
        "sleep_debt_h",
        "Bedtime",
        "Wake Time",
    ]
]

sleep = sleep.dropna(subset=["date"])
sleep = sleep.sort_values("date").reset_index(drop=True)

display(sleep.head())
display(sleep.tail())

,date,sleep_score,Quality,sleep_duration_min,sleep_duration_h,sleep_need_min,sleep_need_h,sleep_debt_h,Bedtime,Wake Time
0,2025-12-25,74.0,Fair,480.0,8.000000,470.0,7.833333,-0.166667,1:41 AM,10:09 AM
1,2025-12-26,84.0,Good,455.0,7.583333,480.0,8.000000,0.416667,1:11 AM,9:00 AM
2,2025-12-27,75.0,Fair,426.0,7.100000,480.0,8.000000,0.900000,1:41 AM,9:11 AM
3,2025-12-28,81.0,Good,526.0,8.766667,500.0,8.333333,-0.433333,12:56 AM,10:00 AM
4,2025-12-29,68.0,Fair,494.0,8.233333,480.0,8.000000,-0.233333,12:54 AM,9:47 AM


,date,sleep_score,Quality,sleep_duration_min,sleep_duration_h,sleep_need_min,sleep_need_h,sleep_debt_h,Bedtime,Wake Time
147,2026-05-21,65.0,Fair,529.0,8.816667,500.0,8.333333,-0.483333,12:35 AM,9:54 AM
148,2026-05-22,78.0,Fair,623.0,10.383333,430.0,7.166667,-3.216667,10:56 PM,9:56 AM
149,2026-05-23,78.0,Fair,390.0,6.500000,500.0,8.333333,1.833333,2:53 AM,9:53 AM
150,2026-05-24,42.0,Poor,287.0,4.783333,420.0,7.000000,2.216667,4:00 AM,9:39 AM
151,2026-05-25,38.0,Poor,197.0,3.283333,540.0,9.000000,5.716667,11:31 AM,2:49 PM


### Notes

On nettoie les données de sommeil.

On convertit la durée de sommeil et le besoin de sommeil en minutes puis en heures.  
On calcule aussi une variable appelée `sleep_debt_h`.

Si `sleep_debt_h` est positif, cela veut dire que la durée de sommeil était inférieure au besoin estimé par Garmin.

In [10]:
stress = stress_raw.copy()

stress["date"] = stress["Date"].apply(parse_garmin_short_date)
stress["avg_stress"] = stress["Avg"].apply(clean_numeric)

stress["rest_min"] = stress["Rest"].apply(duration_to_minutes)
stress["low_stress_min"] = stress["Low"].apply(duration_to_minutes)
stress["medium_stress_min"] = stress["Medium Stress"].apply(duration_to_minutes)
stress["high_stress_min"] = stress["High Stress"].apply(duration_to_minutes)

stress["rest_h"] = stress["rest_min"] / 60
stress["low_stress_h"] = stress["low_stress_min"] / 60
stress["medium_stress_h"] = stress["medium_stress_min"] / 60
stress["high_stress_h"] = stress["high_stress_min"] / 60

stress = stress[
    [
        "date",
        "avg_stress",
        "rest_min",
        "low_stress_min",
        "medium_stress_min",
        "high_stress_min",
        "rest_h",
        "low_stress_h",
        "medium_stress_h",
        "high_stress_h",
    ]
]

stress = stress.dropna(subset=["date"])
stress = stress.sort_values("date").reset_index(drop=True)

display(stress.head())
display(stress.tail())

,date,avg_stress,rest_min,low_stress_min,medium_stress_min,high_stress_min,rest_h,low_stress_h,medium_stress_h,high_stress_h
0,2025-12-24,72.0,0.0,16.0,86.0,82.0,0.000000,0.266667,1.433333,1.366667
1,2025-12-25,34.0,451.0,549.0,142.0,35.0,7.516667,9.150000,2.366667,0.583333
2,2025-12-26,24.0,883.0,341.0,49.0,7.0,14.716667,5.683333,0.816667,0.116667
3,2025-12-27,33.0,519.0,454.0,178.0,8.0,8.650000,7.566667,2.966667,0.133333
4,2025-12-28,35.0,499.0,448.0,183.0,45.0,8.316667,7.466667,3.050000,0.750000


,date,avg_stress,rest_min,low_stress_min,medium_stress_min,high_stress_min,rest_h,low_stress_h,medium_stress_h,high_stress_h
142,2026-05-21,40.0,414.0,408.0,265.0,90.0,6.900000,6.800000,4.416667,1.500000
143,2026-05-22,36.0,434.0,567.0,239.0,32.0,7.233333,9.450000,3.983333,0.533333
144,2026-05-23,40.0,481.0,349.0,227.0,134.0,8.016667,5.816667,3.783333,2.233333
145,2026-05-24,40.0,518.0,336.0,299.0,120.0,8.633333,5.600000,4.983333,2.000000
146,2026-05-25,17.0,848.0,38.0,2.0,0.0,14.133333,0.633333,0.033333,0.000000


### Notes

On nettoie les données de stress.

On garde le stress moyen quotidien et le temps passé dans chaque zone : repos, stress bas, stress moyen et stress élevé.

Ces variables seront utiles pour voir si certains jours combinent mauvais sommeil, stress élevé et fréquence cardiaque au repos plus haute.

In [11]:
body_battery = body_battery_raw.copy()

body_battery["date"] = pd.to_datetime(body_battery["date"], errors="coerce")

body_battery["body_battery_high"] = body_battery["daily_high"].apply(clean_numeric)
body_battery["body_battery_low"] = body_battery["daily_low"].apply(clean_numeric)
body_battery["body_battery_range"] = body_battery["daily_range"].apply(clean_numeric)

body_battery = body_battery[
    [
        "date",
        "body_battery_high",
        "body_battery_low",
        "body_battery_range",
        "extraction_confidence",
        "notes",
    ]
]

body_battery = body_battery.dropna(subset=["date"])
body_battery = body_battery.sort_values("date").reset_index(drop=True)

display(body_battery.head())
display(body_battery.tail())

,date,body_battery_high,body_battery_low,body_battery_range,extraction_confidence,notes
0,2025-12-24,23.0,8.0,15.0,high,NaN
1,2025-12-25,42.0,6.0,36.0,high,NaN
2,2025-12-26,79.0,16.0,63.0,high,NaN
3,2025-12-27,68.0,18.0,50.0,high,NaN
4,2025-12-28,68.0,17.0,51.0,high,NaN


,date,body_battery_high,body_battery_low,body_battery_range,extraction_confidence,notes
148,2026-05-21,53.0,11.0,42.0,high,NaN
149,2026-05-22,56.0,15.0,41.0,high,NaN
150,2026-05-23,60.0,7.0,53.0,high,NaN
151,2026-05-24,20.0,5.0,15.0,high,NaN
152,2026-05-25,97.0,13.0,84.0,high,NaN


### Notes

On nettoie les données de Body Battery.

Ces données viennent des screenshots, donc elles sont estimées et pas aussi fiables qu’un export Garmin brut.  
On garde donc aussi la colonne `extraction_confidence` pour savoir quelles valeurs sont les plus sûres.

In [12]:
activities = activities_raw.copy()

activities["datetime"] = pd.to_datetime(activities["Date"], errors="coerce")
activities["date"] = activities["datetime"].dt.normalize()

activities["distance_km"] = activities["Distance"].apply(clean_numeric)
activities["calories"] = activities["Calories"].apply(clean_numeric)
activities["duration_min"] = activities["Time"].apply(duration_to_minutes)
activities["avg_hr_activity"] = activities["Avg HR"].apply(clean_numeric)
activities["max_hr_activity"] = activities["Max HR"].apply(clean_numeric)
activities["aerobic_te"] = activities["Aerobic TE"].apply(clean_numeric)
activities["steps_activity"] = activities["Steps"].apply(clean_numeric)
activities["body_battery_drain_activity"] = activities["Body Battery Drain"].apply(clean_numeric)

activities_clean = activities[
    [
        "date",
        "datetime",
        "Activity Type",
        "Title",
        "distance_km",
        "calories",
        "duration_min",
        "avg_hr_activity",
        "max_hr_activity",
        "aerobic_te",
        "steps_activity",
        "body_battery_drain_activity",
    ]
]

activities_clean = activities_clean.dropna(subset=["date"])
activities_clean = activities_clean.sort_values("datetime").reset_index(drop=True)

display(activities_clean.head())
display(activities_clean.tail())

,date,datetime,Activity Type,Title,distance_km,calories,duration_min,avg_hr_activity,max_hr_activity,aerobic_te,steps_activity,body_battery_drain_activity
0,1989-12-31,1989-12-31 01:05:14,Running,Running,0.00,84.0,19007.200000,NaN,165.0,2.1,2.147485e+09,-2.0
1,2025-12-27,2025-12-27 00:28:19,Breathwork,Tranquility,NaN,NaN,12.216667,83.0,92.0,0.0,8.000000e+00,NaN
2,2026-01-02,2026-01-02 12:15:22,Walking,Villejuif Walking,1.83,96.0,23.083333,103.0,131.0,0.5,1.878000e+03,-4.0
3,2026-01-02,2026-01-02 12:50:52,Strength Training,Strength,0.00,105.0,31.466667,94.0,120.0,0.3,2.060000e+02,-4.0
4,2026-01-05,2026-01-05 16:22:31,Strength Training,Strength,0.00,241.0,45.333333,118.0,160.0,2.0,1.320000e+02,-6.0


,date,datetime,Activity Type,Title,distance_km,calories,duration_min,avg_hr_activity,max_hr_activity,aerobic_te,steps_activity,body_battery_drain_activity
163,2026-05-21,2026-05-21 19:00:04,Walking,Paris Walking,0.36,32.0,5.238333,129.0,150.0,0.6,498.0,NaN
164,2026-05-23,2026-05-23 10:25:26,Walking,Vélizy-Villacoublay Walking,0.67,60.0,17.750000,95.0,119.0,0.2,1100.0,-1.0
165,2026-05-24,2026-05-24 12:39:14,Walking,Paris Walking,0.51,36.0,10.666667,93.0,111.0,0.2,796.0,NaN
166,2026-05-24,2026-05-24 17:50:04,Walking,Paris Walking,0.85,63.0,16.283333,103.0,119.0,0.4,1292.0,-1.0
167,2026-05-24,2026-05-24 19:07:08,Walking,Paris Walking,0.81,47.0,10.616667,110.0,119.0,0.3,1156.0,-1.0


### Notes

On nettoie les activités sportives.

Ici, chaque ligne correspond à une activité, pas à une journée.  
Pour pouvoir fusionner avec le sommeil, le stress, le heart rate et la Body Battery, il faudra ensuite agréger les activités par jour.

In [13]:
daily_activities = (
    activities_clean
    .groupby("date")
    .agg(
        activity_count=("Activity Type", "count"),
        total_activity_duration_min=("duration_min", "sum"),
        total_activity_distance_km=("distance_km", "sum"),
        total_activity_calories=("calories", "sum"),
        mean_activity_hr=("avg_hr_activity", "mean"),
        max_activity_hr=("max_hr_activity", "max"),
        max_aerobic_te=("aerobic_te", "max"),
        total_activity_steps=("steps_activity", "sum"),
        total_body_battery_drain_activity=("body_battery_drain_activity", "sum"),
    )
    .reset_index()
)

daily_activities["total_activity_duration_h"] = daily_activities["total_activity_duration_min"] / 60

display(daily_activities.head())
display(daily_activities.tail())

,date,activity_count,total_activity_duration_min,total_activity_distance_km,total_activity_calories,mean_activity_hr,max_activity_hr,max_aerobic_te,total_activity_steps,total_body_battery_drain_activity,total_activity_duration_h
0,1989-12-31,1,19007.200000,0.00,84.0,NaN,165.0,2.1,2.147485e+09,-2.0,316.786667
1,2025-12-27,1,12.216667,0.00,0.0,83.0,92.0,0.0,8.000000e+00,0.0,0.203611
2,2026-01-02,2,54.550000,1.83,201.0,98.5,131.0,0.5,2.084000e+03,-8.0,0.909167
3,2026-01-05,2,74.550000,3.21,382.0,114.5,160.0,2.0,1.320000e+02,-10.0,1.242500
4,2026-01-06,2,54.450000,3.03,261.0,106.5,169.0,2.5,1.780000e+02,-8.0,0.907500


,date,activity_count,total_activity_duration_min,total_activity_distance_km,total_activity_calories,mean_activity_hr,max_activity_hr,max_aerobic_te,total_activity_steps,total_body_battery_drain_activity,total_activity_duration_h
84,2026-05-19,2,25.866667,1.64,121.0,112.5,141.0,0.7,2152.0,-2.0,0.431111
85,2026-05-20,1,7.446667,0.42,35.0,113.0,130.0,0.3,602.0,0.0,0.124111
86,2026-05-21,1,5.238333,0.36,32.0,129.0,150.0,0.6,498.0,0.0,0.087306
87,2026-05-23,1,17.750000,0.67,60.0,95.0,119.0,0.2,1100.0,-1.0,0.295833
88,2026-05-24,3,37.566667,2.17,146.0,102.0,119.0,0.4,3244.0,-2.0,0.626111


### Notes

On transforme les activités en résumé quotidien.

Par exemple, si plusieurs marches ou séances sont enregistrées le même jour, on les regroupe pour obtenir :
- le nombre d’activités
- la durée totale
- la distance totale
- les calories totales
- la fréquence cardiaque moyenne pendant les activités
- l’effet aérobie maximal

Cette table pourra être fusionnée avec les autres données quotidiennes.

In [14]:
daily = body_battery.copy()

for df in [hr, sleep, stress, daily_activities]:
    daily = daily.merge(df, on="date", how="outer")

daily = daily.sort_values("date").reset_index(drop=True)

display(daily.head())
display(daily.tail())
print(daily.shape)

,date,body_battery_high,body_battery_low,body_battery_range,extraction_confidence,notes,resting_hr,high_hr,sleep_score,Quality,sleep_duration_min,sleep_duration_h,sleep_need_min,sleep_need_h,sleep_debt_h,Bedtime,Wake Time,avg_stress,rest_min,low_stress_min,medium_stress_min,high_stress_min,rest_h,low_stress_h,medium_stress_h,high_stress_h,activity_count,total_activity_duration_min,total_activity_distance_km,total_activity_calories,mean_activity_hr,max_activity_hr,max_aerobic_te,total_activity_steps,total_body_battery_drain_activity,total_activity_duration_h
0,1989-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,19007.200000,0.0,84.0,NaN,165.0,2.1,2.147485e+09,-2.0,316.786667
1,2025-12-24,23.0,8.0,15.0,high,NaN,88.0,105.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72.0,0.0,16.0,86.0,82.0,0.000000,0.266667,1.433333,1.366667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-12-25,42.0,6.0,36.0,high,NaN,68.0,125.0,74.0,Fair,480.0,8.000000,470.0,7.833333,-0.166667,1:41 AM,10:09 AM,34.0,451.0,549.0,142.0,35.0,7.516667,9.150000,2.366667,0.583333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-12-26,79.0,16.0,63.0,high,NaN,66.0,109.0,84.0,Good,455.0,7.583333,480.0,8.000000,0.416667,1:11 AM,9:00 AM,24.0,883.0,341.0,49.0,7.0,14.716667,5.683333,0.816667,0.116667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-12-27,68.0,18.0,50.0,high,NaN,68.0,139.0,75.0,Fair,426.0,7.100000,480.0,8.000000,0.900000,1:41 AM,9:11 AM,33.0,519.0,454.0,178.0,8.0,8.650000,7.566667,2.966667,0.133333,1.0,12.216667,0.0,0.0,83.0,92.0,0.0,8.000000e+00,0.0,0.203611


,date,body_battery_high,body_battery_low,body_battery_range,extraction_confidence,notes,resting_hr,high_hr,sleep_score,Quality,sleep_duration_min,sleep_duration_h,sleep_need_min,sleep_need_h,sleep_debt_h,Bedtime,Wake Time,avg_stress,rest_min,low_stress_min,medium_stress_min,high_stress_min,rest_h,low_stress_h,medium_stress_h,high_stress_h,activity_count,total_activity_duration_min,total_activity_distance_km,total_activity_calories,mean_activity_hr,max_activity_hr,max_aerobic_te,total_activity_steps,total_body_battery_drain_activity,total_activity_duration_h
149,2026-05-21,53.0,11.0,42.0,high,NaN,64.0,142.0,65.0,Fair,529.0,8.816667,500.0,8.333333,-0.483333,12:35 AM,9:54 AM,40.0,414.0,408.0,265.0,90.0,6.900000,6.800000,4.416667,1.500000,1.0,5.238333,0.36,32.0,129.0,150.0,0.6,498.0,0.0,0.087306
150,2026-05-22,56.0,15.0,41.0,high,NaN,63.0,128.0,78.0,Fair,623.0,10.383333,430.0,7.166667,-3.216667,10:56 PM,9:56 AM,36.0,434.0,567.0,239.0,32.0,7.233333,9.450000,3.983333,0.533333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
151,2026-05-23,60.0,7.0,53.0,high,NaN,63.0,134.0,78.0,Fair,390.0,6.500000,500.0,8.333333,1.833333,2:53 AM,9:53 AM,40.0,481.0,349.0,227.0,134.0,8.016667,5.816667,3.783333,2.233333,1.0,17.750000,0.67,60.0,95.0,119.0,0.2,1100.0,-1.0,0.295833
152,2026-05-24,20.0,5.0,15.0,high,NaN,67.0,131.0,42.0,Poor,287.0,4.783333,420.0,7.000000,2.216667,4:00 AM,9:39 AM,40.0,518.0,336.0,299.0,120.0,8.633333,5.600000,4.983333,2.000000,3.0,37.566667,2.17,146.0,102.0,119.0,0.4,3244.0,-2.0,0.626111
153,2026-05-25,97.0,13.0,84.0,high,NaN,64.0,89.0,38.0,Poor,197.0,3.283333,540.0,9.000000,5.716667,11:31 AM,2:49 PM,17.0,848.0,38.0,2.0,0.0,14.133333,0.633333,0.033333,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


(154, 36)


### Notes

On fusionne toutes les sources dans une seule table quotidienne.

Chaque ligne correspond maintenant à une date.  
Chaque colonne correspond à une métrique Garmin : sommeil, stress, heart rate, Body Battery ou activité.

C’est cette table qui servira de base pour l’EDA puis pour le machine learning.

In [15]:
OUTPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR.mkdir(exist_ok=True)

output_path = OUTPUT_DIR / "daily_health_merged.csv"
daily.to_csv(output_path, index=False)

print(f"Saved merged daily dataset to: {output_path}")

Saved merged daily dataset to: data\processed\daily_health_merged.csv
